# Proyecto: Bank Classifier — EDA + KNN

> **Plataforma:** Databricks Free Edition  
> **Dataset:** `bank.csv` — Campaña de marketing bancario portugués

Las campañas de marketing de los bancos dependen de los datos de los clientes. La cantidad de datos que manejan los bancos es tan grande, que es imposible que un analista saque el máximo partido de forma realmente efectiva en el proceso de toma de decisiones.

Aquí es donde los modelos de aprendizaje automático están ayudando a aumentar drásticamente el rendimiento de estas campañas, al permitir encontrar patrones en los datos que de otro modo pasarían desapercibidos.

## Dataset

El conjunto de datos está relacionado con una campaña de marketing directo de una institución bancaria portuguesa. Durante la campaña (basada en llamadas telefónicas), a menudo se requirió **más de un contacto con el mismo cliente** para ofrecerle la contratación de un depósito bancario a plazo.

**Nuestro objetivo:** predecir si el cliente suscribirá un **depósito a plazo** (`y`), analizando y preprocesando los datos para construir un modelo de clasificación con el algoritmo **k-NN**.

| Variable | Descripción |
|---|---|
| `age` | Edad del cliente |
| `job` | Tipo de trabajo |
| `marital` | Estado civil |
| `education` | Nivel educativo |
| `default` | Crédito en mora |
| `balance` | Saldo medio anual (€) |
| `housing` | Hipoteca contratada |
| `loan` | Crédito personal |
| `contact` | Tipo de contacto |
| `day` | Día del último contacto |
| `month` | Mes del último contacto |
| `duration` | Duración del último contacto (seg.) ⚠️ *data leakage* |
| `campaign` | Nº de contactos en esta campaña |
| `pdays` | Días desde el último contacto previo (`-1` = nunca) |
| `previous` | Nº de contactos anteriores a esta campaña |
| `poutcome` | Resultado de la campaña anterior |
| `y` | **Variable objetivo:** ¿suscribió el depósito? |

---

## Estructura del notebook

| Bloque | Contenido |
|---|---|
| **Bloque 1 — EDA** | Secciones 1 a 5: exploración, visualización y análisis de los datos |
| **Bloque 2 — KNN** | Secciones 6 a 10: preprocesamiento, modelado y evaluación |

Más información: [UCI Machine Learning Repository — Bank Marketing](https://archive.ics.uci.edu/ml/datasets/bank+marketing)

---
# BLOQUE 1 — ANÁLISIS EXPLORATORIO DE DATOS (EDA)
---

## Sección 0 — Importar librerías

Estas son las librerías que necesitaremos a lo largo de todo el proyecto.

In [0]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, classification_report,
    roc_curve, roc_auc_score
)

# Configuración global de gráficos
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams.update({'figure.dpi': 110})

print('✅ Librerías importadas correctamente.')

## Sección 1 — Carga de datos

### 1.1 Cargar el CSV desde Databricks

> **📁 Adaptación Databricks:** En Databricks Free Edition, el fichero `bank.csv` fue subido al **Catalog Explorer** en la ruta  
> `Catalog > workspace > default > bank`.  
> Se accede a él mediante la API de Spark y luego se convierte a Pandas con `.toPandas()`.
>
> Si prefieres cargarlo directamente con Pandas (sin Spark), puedes usar la ruta DBFS que se indica en el comentario alternativo.

In [0]:
# ── Opción A: cargar desde Unity Catalog (recomendado en Databricks) ──────────
# La tabla fue subida en: Catalog Explorer > workspace > default > bank
spark_df = spark.read.table('workspace.default.bank')
data = spark_df.toPandas()

# ── Opción B: cargar desde DBFS si subiste el CSV directamente ────────────────
# data = pd.read_csv('/dbfs/FileStore/bank.csv', sep=';')

# ── Opción C (solo para pruebas rápidas): cargar desde GitHub ─────────────────
# url = 'https://raw.githubusercontent.com/financieras/saturdays_ai/main/bank.csv'
# data = pd.read_csv(url, sep=';')

print(f'✅ Dataset cargado: {data.shape[0]:,} filas × {data.shape[1]} columnas')

## Sección 2 — Exploración general del dataset

### 2.1 Primeras filas

In [0]:
# Muestra las 10 primeras filas
data.head(10)

### 2.2 Dimensiones, tipos y nulos

In [0]:
# Dimensiones
print(f'Filas: {data.shape[0]:,}  |  Columnas: {data.shape[1]}')
print()

# Tipos de dato de cada columna
print('── Tipos de dato ──────────────────────')
print(data.dtypes)
print()

# Valores nulos por columna
print('── Valores nulos por columna ───────────')
nulos = data.isnull().sum()
print(nulos[nulos > 0] if nulos.any() else '✅ Sin valores nulos')

### 2.3 Estadística descriptiva

In [0]:
# Descripción estadística de las variables numéricas
data.describe().round(2)

### 2.4 Frecuencias de las variables categóricas

Mostramos el conteo y porcentaje de cada categoría en las variables de tipo `object`.

In [0]:
cat_cols_raw = data.select_dtypes(include='object').columns.tolist()

for col in cat_cols_raw:
    print(f'--- Variable: {col} ---')
    conteo = data[col].value_counts()
    for val, cnt in conteo.items():
        pct = cnt / len(data) * 100
        print(f'  {str(val):<16} | {cnt:>5} | {pct:>6.1f}%')
    print()

## Sección 3 — Análisis exploratorio de la variable objetivo

### 3.1 Crear copia de trabajo y renombrar la variable objetivo

In [0]:
# Copia profunda para no modificar el dataset original
bank_data = data.copy()

# Renombramos 'y' → 'deposit' para mayor claridad
bank_data = bank_data.rename(columns={'y': 'deposit'})

print('Columnas actuales:')
print(bank_data.columns.tolist())

### 3.2 Codificar la variable objetivo: `yes/no` → `1/0`

In [0]:
# Mapeamos 'yes' → 1, 'no' → 0
bank_data['deposit'] = bank_data['deposit'].map({'yes': 1, 'no': 0})

# Verificamos
print('Conteo de la variable objetivo tras la codificación:')
print(bank_data['deposit'].value_counts())
print(f'Tipo de dato: {bank_data["deposit"].dtype}')

### 3.3 Visualización del desequilibrio de clases

Un punto clave en cualquier proyecto de clasificación es conocer si las clases están **balanceadas**. En este dataset bancario esperamos ver un claro desequilibrio: la mayoría de los clientes NO suscribió el depósito.

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ── Countplot ────────────────────────────────────────────────────────────────
counts = bank_data['deposit'].value_counts()
colores = ['#4C72B0', '#DD8452']
bars = axes[0].bar(['No (0)', 'Sí (1)'], counts.values, color=colores, edgecolor='white', linewidth=0.8)
axes[0].set_title('Distribución de la Variable Objetivo\n', fontsize=14, fontweight='bold')
axes[0].set_xlabel('¿Suscribió el depósito?', fontsize=12)
axes[0].set_ylabel('Número de clientes', fontsize=12)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar, cnt in zip(bars, counts.values):
    pct = cnt / len(bank_data) * 100
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 300,
                 f'{cnt:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=11, fontweight='bold')

# ── Pie chart ────────────────────────────────────────────────────────────────
axes[1].pie(
    counts.values,
    labels=['No suscribió (0)', 'Suscribió (1)'],
    colors=colores,
    autopct='%1.1f%%',
    startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
axes[1].set_title('Proporción de la Variable Objetivo', fontsize=14, fontweight='bold')

plt.suptitle('⚠️  Clases Desequilibradas — 88.3% vs 11.7%', fontsize=13, color='firebrick', y=1.01)
plt.tight_layout()
plt.show()

print('\n📝 Conclusión:')
print('El dataset está fuertemente desbalanceado: casi 9 de cada 10 clientes dijeron «No».')
print('Esto afectará al modelo k-NN, que tenderá a predecir «No» de forma conservadora.')

## Sección 4 — Análisis exploratorio de las variables predictoras

### 4.1 Distribución de la edad (histograma + KDE)

In [0]:
fig, ax = plt.subplots(figsize=(10, 5))

sns.histplot(data=bank_data, x='age', hue='deposit', kde=True,
             palette=['#4C72B0', '#DD8452'], bins=35, alpha=0.55, ax=ax)

ax.axvline(bank_data['age'].mean(), color='red', linestyle='--', lw=1.5,
           label=f"Media global: {bank_data['age'].mean():.1f} años")
ax.axvline(bank_data['age'].median(), color='green', linestyle='-', lw=1.5,
           label=f"Mediana global: {bank_data['age'].median():.0f} años")

ax.set_title('Distribución de la Edad por Grupo (Suscribió / No suscribió)', fontsize=14, fontweight='bold')
ax.set_xlabel('Edad', fontsize=12)
ax.set_ylabel('Frecuencia', fontsize=12)
ax.legend(title='deposit (0=No, 1=Sí)')
plt.tight_layout()
plt.show()

### 4.2 Gráfico de violín — Edad vs. Depósito

El violín muestra la distribución completa de la edad para cada grupo: la anchura refleja la densidad de clientes en cada rango de edad.

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── Violín: Edad por depósito ─────────────────────────────────────────────────
sns.violinplot(
    data=bank_data, x='deposit', y='age', hue='deposit',
    palette=['#4C72B0', '#DD8452'], inner='quartile',
    legend=False, ax=axes[0]
)
axes[0].set_title('Distribución de Edad por Decisión de Depósito', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Depósito (0=No, 1=Sí)', fontsize=11)
axes[0].set_ylabel('Edad', fontsize=11)
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['No suscribió', 'Suscribió'])

# ── Violín: Balance por depósito (limitado a percentil 95 para evitar outliers) ─
p95 = bank_data['balance'].quantile(0.95)
subset = bank_data[bank_data['balance'] <= p95]
sns.violinplot(
    data=subset, x='deposit', y='balance', hue='deposit',
    palette=['#4C72B0', '#DD8452'], inner='quartile',
    legend=False, ax=axes[1]
)
axes[1].set_title('Distribución de Saldo por Decisión de Depósito\n(hasta percentil 95)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Depósito (0=No, 1=Sí)', fontsize=11)
axes[1].set_ylabel('Saldo (€)', fontsize=11)
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(['No suscribió', 'Suscribió'])

plt.suptitle('Gráficos de Violín — Variables Numéricas Clave', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('\n📝 Lectura rápida:')
print('  - Las líneas internas del violín marcan los cuartiles (Q1, mediana, Q3).')
print('  - Un violín más ancho en la zona alta de edad/balance indica mayor concentración de clientes ahí.')

### 4.3 Barplots de variables categóricas vs. tasa de suscripción

Para cada variable categórica creamos un barplot cuya altura representa la **tasa de éxito** (proporción de clientes que sí suscribieron). Esto permite identificar los segmentos más y menos receptivos a la campaña.

In [0]:
cat_features = [col for col in bank_data.columns
                if bank_data[col].dtype == 'object' and col != 'deposit']

n_cols = 3
n_rows = int(np.ceil(len(cat_features) / n_cols))
fig, axes = plt.subplots(nrows=n_rows, ncols=n_cols, figsize=(20, n_rows * 5))
axes = axes.flat

for i, col in enumerate(cat_features):
    tasa = bank_data.groupby(col)['deposit'].mean().sort_values(ascending=False)
    sns.barplot(x=tasa.index, y=tasa.values, ax=axes[i], palette='magma')
    axes[i].axhline(bank_data['deposit'].mean(), color='red', linestyle='--', lw=1.2,
                    label=f"Media global ({bank_data['deposit'].mean():.1%})")
    axes[i].set_title(f'Tasa de suscripción por {col}', fontsize=13, fontweight='bold')
    axes[i].set_ylabel('Tasa de éxito', fontsize=11)
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=40)
    axes[i].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    axes[i].legend(fontsize=9)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Tasa de Suscripción del Depósito por Categoría\n(línea roja = media global ~11.7%)',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 4.4 Pairplot de variables numéricas clave

El pairplot muestra la distribución individual (diagonal) y las relaciones entre pares de variables (resto de la cuadrícula), diferenciando por color si el cliente suscribió o no el depósito.

> **💡 Nota de rendimiento:** Para evitar tiempos de espera excesivos con 45 000 filas, tomamos una muestra aleatoria estratificada de 5 000 registros.

In [0]:
import warnings

cols_pairplot = ['age', 'balance', 'campaign', 'previous', 'deposit']

sample = (
    bank_data[cols_pairplot]
    .groupby('deposit', group_keys=False)
    .apply(lambda g: g.sample(min(len(g), 2500), random_state=42))
)

sample_plot = sample.copy()
sample_plot['deposit'] = sample_plot['deposit'].map({0: 'No suscribió', 1: 'Suscribió'})

with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    g = sns.pairplot(
        sample_plot,
        hue='deposit',
        hue_order=['No suscribió', 'Suscribió'],
        palette={'No suscribió': '#4C72B0', 'Suscribió': '#DD8452'},
        corner=True,
        diag_kind='kde',
        plot_kws={'alpha': 0.35, 's': 18, 'linewidths': 0},
        diag_kws={'fill': True, 'alpha': 0.4}
    )

g.figure.suptitle(
    'Pairplot — Variables Numéricas Clave (muestra n=5 000)',
    fontsize=15, fontweight='bold', y=1.02
)

labels_map = {'age': 'Edad', 'balance': 'Saldo (€)', 'campaign': 'Contactos\ncampaña',
              'previous': 'Contactos\nprevios'}
for ax in g.axes.flatten():
    if ax is None:
        continue
    xl = ax.get_xlabel()
    yl = ax.get_ylabel()
    if xl in labels_map:
        ax.set_xlabel(labels_map[xl], fontsize=9)
    if yl in labels_map:
        ax.set_ylabel(labels_map[yl], fontsize=9)

plt.show()

## Sección 5 — Consultas analíticas al dataset

En esta sección pasamos de *describir* los datos a *generar conocimiento accionable* para el banco.

### 5.1 ¿Cuántos clientes NO han sido contactados nunca?

In [0]:
# pdays == -1 significa que el cliente no fue contactado en campañas previas
no_contactados = (bank_data['pdays'] == -1).sum()
pct = no_contactados / len(bank_data) * 100

print(f'Clientes sin contacto previo: {no_contactados:,}  ({pct:.1f}% del total)')
print()
print('📝 Más del 80% de los clientes son "nuevos" para la campaña: no tienen historial previo.')
print('   El modelo tendrá que basarse en su perfil demográfico para hacer predicciones sobre ellos.')

### 5.2 De los contactados previamente, ¿cuántos días han pasado desde el más antiguo?

In [0]:
contactados_previos = bank_data[bank_data['pdays'] != -1]

max_dias = contactados_previos['pdays'].max()
print(f'Días desde el contacto más antiguo: {max_dias} días ({max_dias / 365:.1f} años aprox.)')
print()
print('Estadística descriptiva de pdays (solo contactados):')
print(contactados_previos['pdays'].describe().round(1))

### 5.3 ¿Qué perfiles de trabajo tienen una tasa de éxito superior al 25%?

In [0]:
tasa_por_job = bank_data.groupby('job')['deposit'].mean().sort_values(ascending=False)
top_jobs = tasa_por_job[tasa_por_job > 0.25]

print('Trabajos con tasa de éxito > 25%:')
for job, tasa in top_jobs.items():
    print(f'  {job:<18} → {tasa:.1%}')

print(f'\n📝 Comparativa: la tasa global es {bank_data["deposit"].mean():.1%}. Los estudiantes duplican esa tasa.')

### 5.4 ¿Cuántos clientes con historial de impago (default) suscribieron el depósito?

In [0]:
# En este punto 'default' aún es texto (yes/no), usamos el valor original
deudores_exitosos = bank_data[(bank_data['default'] == 'yes') & (bank_data['deposit'] == 1)]
total_deudores = (bank_data['default'] == 'yes').sum()

print(f'Clientes con default=yes: {total_deudores:,}')
print(f'De ellos, suscribieron el depósito: {len(deudores_exitosos)} ({len(deudores_exitosos)/total_deudores:.1%})')
print()
print('📝 Recomendación: el retorno de invertir recursos en clientes con impago es muy bajo.')
print('   Priorizar clientes sin historial de impago mejora la eficiencia de la campaña.')

### 5.5 Balance medio por nivel educativo y resultado de la campaña

In [0]:
analisis_balance = (
    bank_data.groupby(['education', 'deposit'])['balance']
    .mean()
    .unstack()
    .round(0)
    .astype(int)
)
analisis_balance.columns = ['No suscribió (€)', 'Suscribió (€)']
analisis_balance['Diferencia (€)'] = analisis_balance['Suscribió (€)'] - analisis_balance['No suscribió (€)']

print('Balance promedio por nivel educativo y resultado:')
print(analisis_balance)

print()
print('📝 En todos los niveles educativos, quienes suscribieron tienen un saldo medio')
print('   significativamente más alto (+400 a +800 €). El segmento tertiary es el más rentable.')

### 5.6 Técnicos que suscribieron el depósito

In [0]:
tecnicos_exito = bank_data[(bank_data['job'] == 'technician') & (bank_data['deposit'] == 1)]

print(f'Técnicos que suscribieron el depósito: {len(tecnicos_exito):,}')
print(f'Edad media de este grupo: {tecnicos_exito["age"].mean():.1f} años')
print()
print('Primeras filas:')
display(tecnicos_exito[['age', 'job', 'marital', 'education', 'balance', 'deposit']].head())

---
# BLOQUE 2 — MODELADO CON K-NN
---

## Sección 6 — Preprocesamiento de datos

Antes de alimentar el algoritmo k-NN, necesitamos convertir todas las variables a formato numérico y eliminar la información que podría sesgar el modelo.

### 6.1 Transformar meses de texto a número

In [0]:
meses_dict = {
    'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4, 'may': 5, 'jun': 6,
    'jul': 7, 'aug': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12
}

print('Conteo ANTES:')
print(bank_data['month'].value_counts().sort_index())

bank_data['month'] = bank_data['month'].map(meses_dict)

print('\nConteo DESPUÉS:')
print(bank_data['month'].value_counts().sort_index())
print(f'\nNuevo tipo: {bank_data["month"].dtype}')

### 6.2 Codificar variables binarias: `default`, `housing`, `loan` → `0/1`

In [0]:
mapping_binario = {'yes': 1, 'no': 0}
cols_binarias = ['default', 'housing', 'loan']

for col in cols_binarias:
    bank_data[col] = bank_data[col].map(mapping_binario)

print('Verificación — primeras 5 filas de las columnas transformadas:')
print(bank_data[cols_binarias].head())

print('\nConteo de valores únicos por columna:')
for col in cols_binarias:
    print(f'  {col.upper()}: {dict(bank_data[col].value_counts())}')

### 6.3 Unificar categorías ambiguas en `poutcome`

La categoría `other` de `poutcome` no tiene una interpretación clara, así que la agrupamos dentro de `unknown` para simplificar el modelo.

In [0]:
bank_data['poutcome'] = bank_data['poutcome'].replace('other', 'unknown')

conteo_poutcome = bank_data['poutcome'].value_counts()
print('Distribución de poutcome (tras unificación other → unknown):')
for val, cnt in conteo_poutcome.items():
    print(f'  {val:<12} → {cnt:>6,}  ({cnt/len(bank_data):.1%})')

### 6.4 Eliminar variables problemáticas: `pdays`, `contact`, `duration`

| Columna | Motivo de eliminación |
|---|---|
| `pdays` | El -1 masivo (81.7%) la hace casi binaria y con escasa información real |
| `contact` | Baja relevancia; más del 28% son `unknown` |
| `duration` | **Data leakage**: la duración de la llamada se conoce solo *después* de realizarla |

In [0]:
cols_eliminar = ['pdays', 'contact', 'duration']
bank_data = bank_data.drop(columns=cols_eliminar)

print(f'Columnas eliminadas: {cols_eliminar}')
print(f'Columnas restantes ({len(bank_data.columns)}): {bank_data.columns.tolist()}')

### 6.5 One-Hot Encoding de variables categóricas restantes

El algoritmo k-NN calcula distancias geométricas, por lo que necesita que **todos** los datos sean numéricos. Aplicamos `get_dummies()` a las variables categóricas restantes (`job`, `marital`, `education`, `poutcome`).

In [0]:
cat_cols_ohe = bank_data.select_dtypes(include='object').columns.tolist()
print(f'Columnas a codificar: {cat_cols_ohe}')

bank_data_final = pd.get_dummies(bank_data, columns=cat_cols_ohe, drop_first=True)

# Convertir columnas booleanas a entero (compatibilidad con sklearn)
bool_cols = bank_data_final.select_dtypes(include='bool').columns
bank_data_final[bool_cols] = bank_data_final[bool_cols].astype(int)

print(f'\nForma original:          {bank_data.shape}')
print(f'Forma tras One-Hot:      {bank_data_final.shape}')
print(f'\nColumnas nuevas creadas: {bank_data_final.shape[1] - bank_data.shape[1] + len(cat_cols_ohe)}')
print()
bank_data_final.head(3)

### 6.6 Matriz de correlación

Ahora que el dataset es completamente numérico, calculamos la correlación de Pearson para identificar las variables más relacionadas con `deposit`.

In [0]:
corr_matrix = bank_data_final.corr()

plt.figure(figsize=(18, 12))
sns.heatmap(
    corr_matrix,
    annot=False,
    cmap='coolwarm',
    center=0,
    linewidths=0.4,
    vmin=-0.5, vmax=0.5
)
plt.title('Matriz de Correlación del Dataset Final (tras One-Hot Encoding)', fontsize=15, fontweight='bold')
plt.xticks(fontsize=8, rotation=45, ha='right')
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

# Variables más correlacionadas con el target
corr_target = corr_matrix['deposit'].drop('deposit').sort_values(ascending=False)
print('\nTop 6 variables correlacionadas con ÉXITO (deposit=1):')
print(corr_target.head(6).to_string())
print('\nTop 5 variables correlacionadas con FRACASO (deposit=0):')
print(corr_target.tail(5).to_string())

## Sección 7 — División del dataset en entrenamiento y prueba

### 7.1 Separar features (X) y target (y)

In [0]:
y = bank_data_final['deposit']
X = bank_data_final.drop('deposit', axis=1)

print(f'X (características): {X.shape[0]:,} filas × {X.shape[1]} columnas')
print(f'y (objetivo):        {y.shape[0]:,} registros')
print(f'Proporción de positivos en y: {y.mean():.1%}')

### 7.2 División 75% entrenamiento / 25% prueba

- `random_state=42` → reproducibilidad: la misma división siempre.
- `stratify=y` → garantiza que ambos conjuntos mantengan la misma proporción de clases (~11.7% de `1s`).

In [0]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print(f'Muestras de entrenamiento: {X_train.shape[0]:,}  ({X_train.shape[0]/len(X):.0%})')
print(f'Muestras de prueba:        {X_test.shape[0]:,}   ({X_test.shape[0]/len(X):.0%})')
print()
print(f'Proporción de positivos — train: {y_train.mean():.1%}  |  test: {y_test.mean():.1%}')
print('✅ La estratificación mantiene la proporción de clases en ambos conjuntos.')

## Sección 8 — Normalización de datos

k-NN es **sensible a la escala**: sin normalización, variables como `balance` (rango: -8.000 a 102.000) dominarían sobre las variables binarias (0/1). El `MinMaxScaler` lleva todas las variables al rango [0, 1].

> ⚠️ El escalador se **ajusta** (`fit`) solo con los datos de **entrenamiento** para evitar data leakage. Los datos de prueba se transforman con los parámetros ya aprendidos.

In [0]:
scaler = MinMaxScaler()

# fit_transform en train: aprende min/max y transforma
X_train_scaled = scaler.fit_transform(X_train)

# solo transform en test: usa los min/max aprendidos de train
X_test_scaled = scaler.transform(X_test)

print('✅ Normalización completada.')
print(f'Rango de X_train_scaled: [{X_train_scaled.min():.4f}, {X_train_scaled.max():.4f}]')
print(f'Rango de X_test_scaled:  [{X_test_scaled.min():.4f}, {X_test_scaled.max():.4f}]')

## Sección 9 — Entrenamiento del modelo k-NN

### 9.1 Búsqueda del valor óptimo de k (k = 1 a 20)

In [0]:
import os
import sys
import warnings
warnings.filterwarnings("ignore")

valores_k = range(1, 21)
precisiones = []

# Redirigir stderr para silenciar el ruido de threadpoolctl
stderr_original = sys.stderr
sys.stderr = open(os.devnull, 'w')

# Ejecutar el bucle sin ruido
for k in valores_k:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    y_pred_k = knn.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred_k)
    precisiones.append(acc)

# Restaurar stderr
sys.stderr.close()
sys.stderr = stderr_original

# Mostramos todos los resultados al final, limpios
print("=" * 35)
for k, acc in zip(valores_k, precisiones):
    print(f'  k = {k:>2}: Accuracy = {acc:.4f}')

mejor_k = list(valores_k)[precisiones.index(max(precisiones))]
mejor_acc = max(precisiones)
print("=" * 35)
print(f'\n✅ MEJOR k: {mejor_k}  →  Accuracy = {mejor_acc:.4f}')

### 9.2 Gráfico de Accuracy y Tasa de Error vs. k

In [0]:
tasa_error = [1 - acc for acc in precisiones]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# ── Accuracy ─────────────────────────────────────────────────────────────────
ax1.plot(list(valores_k), precisiones, color='#2ca02c', linestyle='--',
         marker='o', markerfacecolor='#ff7f0e', markersize=8, lw=2)
ax1.axvline(mejor_k, color='firebrick', linestyle=':', lw=1.5, label=f'k óptimo = {mejor_k}')
ax1.set_title('Evolución del Accuracy vs. Valor de k', fontsize=13, fontweight='bold')
ax1.set_xlabel('Valor de k', fontsize=12)
ax1.set_ylabel('Accuracy', fontsize=12)
ax1.set_xticks(list(valores_k))
ax1.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax1.legend()
ax1.grid(True, alpha=0.35)

# ── Tasa de Error ─────────────────────────────────────────────────────────────
ax2.plot(list(valores_k), tasa_error, color='#d62728', linestyle='--',
         marker='o', markerfacecolor='#1f77b4', markersize=8, lw=2)
ax2.axvline(mejor_k, color='firebrick', linestyle=':', lw=1.5, label=f'k óptimo = {mejor_k}')
ax2.set_title('Evolución de la Tasa de Error vs. Valor de k', fontsize=13, fontweight='bold')
ax2.set_xlabel('Valor de k', fontsize=12)
ax2.set_ylabel('Tasa de Error', fontsize=12)
ax2.set_xticks(list(valores_k))
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax2.legend()
ax2.grid(True, alpha=0.35)

plt.suptitle('Búsqueda del Hiperparámetro Óptimo k (k-NN)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print('📝 El codo del error se produce entre k=1 y k=4.')
print('   A partir de k≈10, la mejora es marginal: el modelo ha encontrado su equilibrio.')

### 9.3 Entrenamiento del modelo final

Elegimos **k impar** para evitar empates en la votación mayoritaria. Si el k óptimo es par, tomamos el siguiente impar de precisión comparable.

In [0]:
# Si el mejor k es par, tomamos el siguiente impar con precisión similar
k_final = mejor_k if mejor_k % 2 != 0 else mejor_k + 1
print(f'k óptimo encontrado: {mejor_k}  →  k final elegido (impar): {k_final}')
print(f'Accuracy con k={k_final}: {precisiones[k_final - 1]:.4f}')
print()

# Entrenamiento del modelo final
knn_final = KNeighborsClassifier(n_neighbors=k_final)
knn_final.fit(X_train_scaled, y_train)

print(f'✅ Modelo k-NN entrenado con k = {k_final}')

## Sección 10 — Evaluación del modelo

### 10.1 Predicciones sobre el conjunto de prueba

In [0]:
import os
import sys
import warnings
warnings.filterwarnings("ignore")

# Redirigir stderr para silenciar el ruido de threadpoolctl
stderr_original = sys.stderr
sys.stderr = open(os.devnull, 'w')

# Ejecutar las llamadas a sklearn
y_pred = knn_final.predict(X_test_scaled)
acc_train = accuracy_score(y_train, knn_final.predict(X_train_scaled))
acc_test  = accuracy_score(y_test, y_pred)

# Restaurar stderr
sys.stderr.close()
sys.stderr = stderr_original

# Ahora los resultados aparecen solos, sin ruido encima
print("=" * 45)
print(f'Accuracy en entrenamiento: {acc_train:.4f}  ({acc_train:.1%})')
print(f'Accuracy en prueba:        {acc_test:.4f}  ({acc_test:.1%})')
print()

if abs(acc_train - acc_test) < 0.02:
    print('✅ Diferencia < 2%: el modelo generaliza bien.')
else:
    print('⚠️  Diferencia > 2%: posible overfitting.')

print("=" * 45)

comparativa = pd.DataFrame({'Valor Real': y_test.values, 'Predicción': y_pred})
print('\nPrimeras 10 predicciones vs. valores reales:')
print(comparativa.head(10).to_string(index=False))

### 10.2 Matriz de confusión

In [0]:
matriz = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = matriz.ravel()

print(f'--- Matriz de Confusión (k={k_final}) ---')
print(f'  Verdaderos Negativos (TN): {tn:>5,}  — No suscribió y el modelo dijo «No» ✅')
print(f'  Falsos Positivos      (FP): {fp:>5,}  — No suscribió pero el modelo dijo «Sí» ❌ (coste: llamada inútil)')
print(f'  Falsos Negativos      (FN): {fn:>5,}  — Suscribió pero el modelo dijo «No» ❌ (coste: oportunidad perdida)')
print(f'  Verdaderos Positivos  (TP): {tp:>5,}  — Suscribió y el modelo dijo «Sí» ✅')
print()

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=matriz, display_labels=['No Depósito', 'Depósito'])
disp.plot(cmap='Blues', values_format='d', ax=ax)
ax.set_title(f'Matriz de Confusión (k={k_final})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n📝 El modelo es excelente descartando clientes (TN muy alto),')
print(f'   pero pierde {fn:,} oportunidades de venta reales (FN elevado).')
print('   Esto es consecuencia del fuerte desequilibrio de clases (88.3% vs 11.7%).')

### 10.3 Informe de clasificación

In [0]:
print(f'=== Informe de Clasificación (k={k_final}) ===')
print(classification_report(y_test, y_pred, target_names=['No Depósito', 'Depósito']))

print()
print('📝 Conclusiones del informe:')
print('  - «No Depósito»: Precision ~90%, Recall ~99% → el modelo casi nunca falla descartando.')
print('  - «Depósito»:    Precision ~60%, Recall ~17% → solo detecta 1 de cada 6 suscriptores reales.')
print('  - El Accuracy global (~89%) es engañoso: refleja el desequilibrio, no la capacidad real del modelo.')
print('  - El F1-score de la clase positiva (~26%) es el indicador más honesto del rendimiento.')

### 10.4 Curva ROC y AUC

In [0]:
# Probabilidades de la clase positiva (1: Depósito)
y_probs = knn_final.predict_proba(X_test_scaled)[:, 1]

# Coordenadas de la curva y valor AUC
fpr, tpr, thresholds = roc_curve(y_test, y_probs)
auc_value = roc_auc_score(y_test, y_probs)

print(f'AUC = {auc_value:.4f}')
print()

fig, ax = plt.subplots(figsize=(8, 6))

ax.fill_between(fpr, tpr, alpha=0.12, color='darkorange')
ax.plot(fpr, tpr, color='darkorange', lw=2.5,
        label=f'Curva ROC  (AUC = {auc_value:.4f})')
ax.plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--',
        label='Referencia aleatoria (AUC = 0.50)')

ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('Tasa de Falsos Positivos (1 − Especificidad)', fontsize=12)
ax.set_ylabel('Tasa de Verdaderos Positivos (Recall / Sensibilidad)', fontsize=12)
ax.set_title(f'Curva ROC — Clasificador k-NN (k={k_final})', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print('\n📝 Interpretación del AUC:')
print('  - AUC = 1.0 → Clasificador perfecto.')
print('  - AUC = 0.5 → No mejor que el azar (línea punteada azul).')
print(f'  - AUC = {auc_value:.2f} → El modelo tiene capacidad de discriminación moderada-buena.')
print()
print('📝 Conclusión final del proyecto:')
print('   El modelo es muy fiable para descartar (Recall del «No» ~99%), pero conservador')
print('   para capturar oportunidades de venta. Con un AUC sólido, el potencial existe:')
print('   bajar el umbral de probabilidad (de 0.5 a ~0.3) permitiría captar más clientes')
print('   a costa de algunas llamadas extra. No hay que cambiar el algoritmo, sino ajustar el umbral.')